# Golfo Mexico — Datamatch · Threshold Analysis
**Date:** 2026-07-23  
**File:** `Datamatch_Femsa_Nicaragua_Final_Complete_20260522.csv`

Workflow:
1. Run **Section 2–4** (load data + explore threshold samples)
2. Decide your threshold visually
3. Set `THRESHOLD` in **Section 5** → run to generate the boolean `super_match` column
4. **Section 6** exports the enriched CSV

## 1. Imports

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 45)
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ Ready')

✅ Ready


## 2. Load Data

In [2]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
CSV_FILE = os.path.join(NOTEBOOK_DIR, 'final_output.csv')

df = pd.read_csv(CSV_FILE)

matched = df[df['total_score'].notna()].copy()

print(f'Total rows      : {len(df):,}')
print(f'With total_score: {len(matched):,}')
print(f'No score (NaN)  : {df["total_score"].isna().sum():,}')
print()
print('total_score stats:')
print(matched['total_score'].describe().to_string())

Total rows      : 6,860
With total_score: 331
No score (NaN)  : 6,529

total_score stats:
count   331.0000
mean      0.8728
std       0.0831
min       0.7003
25%       0.8136
50%       0.8705
75%       0.9655
max       0.9995


In [3]:
#Se renombra las columnas
df = df.rename(columns={'id.c':'dataplor_id','name.c':'dataplor_name',
                                           'address.c':'dataplor_address','latitude.c':'dataplor_latitude',
                                           'longitude.c':'dataplor_longitude','id.o':'pos_id',
                                           'name.o':'pos_name','address.o':'pos_address',
                                           'latitude.o':'pos_latitude','longitude.o':'pos_longitude'})
matched = df[df['total_score'].notna()].copy()

## 3. Columns for Visual Validation

Only the fields you need to judge whether two records are the same real-world point:

In [6]:
DISPLAY_COLS = [
    'dataplor_name',      # POI name (Dataplor side)
    'pos_name',           # POI name (POS side)
    'dataplor_address',   # Address (Dataplor)
    #'pos_address',        # Address (POS)
    'distance_score',     # Geographic proximity (0–1)
    'name_score',         # String similarity of names (0–1)
    'address_score',      # String similarity of addresses (0–1)
    'phonetic_score',     # Phonetic similarity (0–1)
    'total_score',        # Weighted composite (this drives super_match)
    'distance',       # Geographic distance in kilometers
]

print('Display columns set:', DISPLAY_COLS)

Display columns set: ['dataplor_name', 'pos_name', 'dataplor_address', 'distance_score', 'name_score', 'address_score', 'phonetic_score', 'total_score', 'distance']


In [7]:
matched.head()[DISPLAY_COLS]

,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
153,Vig’s Pollo,VIGS POLLO TEJERIA,"Carr. Xalapa Veracruz, Las Bajadas",0.9446,0.8979,0.0000,0.7500,0.9139,0.0138
166,Abarrotes El Cristo,EL CRISTO,Colonia Renacimiento,0.9656,0.8219,0.0000,0.5700,0.8811,0.0086
195,Abarrotes Camila,AB CAMILA,C. San Luciano 540,0.9522,0.6755,0.0000,0.8000,0.8200,0.0120
197,Cremería Blanquita,CREMERIA BLANQUITA,"Portal, Fraccionamiento Geovillas los Pinos",0.9735,1.0000,0.0000,1.0000,0.9867,0.0066
205,Abarrotes Valeria,VALERIA,P.º Oasis,0.9346,0.8219,0.0000,0.5000,0.8621,0.0163


## 4. Threshold Explorer — 0.95 → 0.40

Each block shows the **10 records with the lowest `total_score` that still clear the threshold** — i.e. the most marginal cases at that cut-off.  
If they look like the same place → that threshold is safe.  
If some look wrong → the threshold is too low, go one step up.

In [8]:
from IPython.display import display, HTML

SCORE_COLS = ['distance_score', 'name_score', 'address_score', 'phonetic_score', 'total_score']

def style_sample(df_sub, thr):
    fmt = {c: '{:.4f}' for c in SCORE_COLS if c in df_sub.columns}
    return (
        df_sub.style
        .format(fmt, na_rep='—')
        .background_gradient(subset=['total_score'], cmap='RdYlGn', vmin=0.40, vmax=0.95)
        .background_gradient(subset=['distance_score','name_score','address_score','phonetic_score'],
                             cmap='RdYlGn', vmin=0, vmax=1)
        .set_properties(**{
            'font-size': '11px',
            'border': '1px solid #ddd',
            'padding': '4px 8px',
            'white-space': 'nowrap',
            'max-width': '230px',
            'overflow': 'hidden',
            'text-overflow': 'ellipsis',
        })
        .set_table_styles([
            {'selector': 'th', 'props': [
                ('background-color', '#1a1a2e'), ('color', 'white'),
                ('font-size', '11px'), ('padding', '6px 8px'),
            ]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', "#c07e7e")]},
            {'selector': 'tr:hover',           'props': [('background-color', "#a0811b")]},
        ])
    )

thresholds = [round(t, 2) for t in np.arange(0.95, 0.39, -0.05)]

for thr in thresholds:
    above = matched[matched['total_score'] >= thr]
    sample = above.nsmallest(10, 'total_score')[DISPLAY_COLS].reset_index(drop=True)
    n_qualify = len(above)

    header = (
        f"<h3 style='margin-top:32px; color:#1a1a2e; "
        f"border-bottom:3px solid #e94560; padding-bottom:5px;'>"
        f"Threshold &ge; <span style='color:#e94560'>{thr:.2f}</span>"
        f" &nbsp;&mdash;&nbsp; "
        f"<span style='font-weight:normal; font-size:14px; color:#555'>"
        f"{n_qualify:,} records qualify &nbsp;|&nbsp; "
        f"showing 10 with lowest total_score</span></h3>"
    )
    display(HTML(header))
    display(style_sample(sample, thr))

,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,Abarrotes Las Negritas,ABARROTES LAS NEGRITAS,Rio Medellín 29,0.9025,1.0000,0.0000,1.0000,0.9512,0.024386
1,Asadero Dario Tejería,ASADERO DARIO TEJERIA,"En La, Abedules 80, LT 1",0.9032,1.0000,0.0000,1.0000,0.9516,0.024205
2,Carnicería lulu,CARNICERIA LULU,Jinicuil 94,0.9073,1.0000,0.0000,1.0000,0.9536,0.023178
3,Abarrotes Emma,ABARROTES EMMA,Tlapacoyan,0.9082,1.0000,0.0000,1.0000,0.9541,0.022945
4,MAR & TIERRA,MAR Y TIERRA,Carr. a dos Lomas 430,0.9112,1.0000,0.0000,1.0000,0.9556,0.022209
5,3 Carabelas,3 CARABELAS,"Agua Marina MZ 24 LT 5-A, entre Gema y coral, Las Bajadas",0.9127,1.0000,0.0000,1.0000,0.9563,0.021836
6,Abarrotes Sacramento,ABARROTES SACRAMENTO,—,0.9139,1.0000,0.0000,1.0000,0.9570,0.021521
7,Abarrotes La Buena Nueva,ABARROTES LA BUENA NUEVA,Cacao 1,0.9213,1.0000,0.0000,1.0000,0.9607,0.019668
8,Abarrotes Mariam,ABARROTES MARIAM,Río Colorado 155,0.9225,1.0000,0.0000,1.0000,0.9612,0.019386
9,Cremería AURE,CREMERIA AURE,San Pedro y San Pablo 293,0.9233,1.0000,0.0000,1.0000,0.9617,0.019170


,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,Abarrotes y Pollería ROSY,POLLERIA ROSY,C. Leonardo Pasquel 255,0.9313,0.8979,0.0000,0.6700,0.9032,0.017167
1,Abts. San Gabriel,ABARROTES SAN GABRIEL,Coco 18,0.9891,0.8063,0.0000,0.9600,0.9054,0.002722
2,Mariscos La Palapa,MARISCOS LA PALAPA EXPRES,Carr. a Mata de Pita 2279,0.9356,0.8979,0.0000,0.7200,0.9079,0.016106
3,Doña Pau,COMEDOR DOÑA PAU,"Adalberto Tejeda Lt. 2, Mata de Pita",0.9470,0.8979,0.0000,0.6200,0.9086,0.013256
4,Pasticheff 2,ABARROTES PASTI-CHEFF 2,Cto. Abeto Rojo 32-54,0.9492,0.8979,0.0000,0.6200,0.9097,0.012712
5,Pollería y Verdulería Gadiel,POLLERIA GADIEL,Zapote 11,0.9451,0.8979,0.0000,0.7000,0.9116,0.013733
6,Abarrotes Meleke de Oro,SUPER MELEKE DE ORO,C. Río Besaya 131,0.9486,0.8979,0.0000,0.6700,0.9119,0.012844
7,Vig’s Pollo,VIGS POLLO TEJERIA,"Carr. Xalapa Veracruz, Las Bajadas",0.9446,0.8979,0.0000,0.7500,0.9139,0.013839
8,Cremeria dorado,ABARROTES Y CREMERIA EL DORADO,Carr. a dos Lomas,0.9652,0.8979,0.0000,0.7300,0.9232,0.008691
9,Abarrotes Blanca Flor,BLANCA FLOR,P.º Mixtli 214,0.9653,0.8979,0.0000,0.7300,0.9232,0.008686


,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,Abarrotes Benny,ABARROTES BENY,"privada laguna de lagarto, Av. Sotavento",0.9959,0.6755,0.0000,1.0000,0.8519,0.001017
1,ABARROTES PATTY,ABARROTES PATY,Av Río Medio 1612,0.9968,0.6755,0.0000,1.0000,0.8524,0.000800
2,Abarrotes TADEO,TADEO,"casi esquina, Brillante, Plata 25",0.9257,0.8219,0.0000,0.4000,0.8527,0.018578
3,Cremería y Salchichonería Juquilita,CREMERIA JUQUILITA,Av Río Medio 52a,0.8271,0.8979,0.0000,0.7200,0.8536,0.043224
4,Proveedora de Alimentos Gigio - Lomas IV,PROVEEDORA DE ALIMENTOS GIGIO,C. Río Acre 249-262,0.8505,0.8703,0.0000,0.8100,0.8574,0.037387
5,La parroquia,CARNICERIA LA PARROQUIA,San Marcos 333,0.9258,0.8219,0.0000,0.5000,0.8578,0.018542
6,Abarrotes Sandra Stephanie,ABARROTES SANDRA STEPHANY,"Río Besaya 651, Lomas de Río Medio IV",0.8907,0.8063,0.0000,1.0000,0.8582,0.027323
7,ABARROTES los manguitos,LOS MANGUITOS,Columna 4242,0.9167,0.8219,0.0000,0.6200,0.8592,0.020824
8,Abarrotes Del campo,EL CAMPO,C. Gaviotas 23,0.9309,0.8219,0.0000,0.5000,0.8603,0.017274
9,La Casa del Volovan,CASA DEL VOLOVAN PATIO TEJERIA,"Carretera Federal Veracruz, Plaza Patio, Xalapa Local S15",0.9154,0.8219,0.0000,0.6700,0.8611,0.021141


,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,Abarrotes Super Jana,MINISUPER JANA,Araucarias 15,0.9753,0.6755,0.0000,0.2500,0.8041,0.006166
1,Abarrotes jacky,HCV JACKY,Barbiquejo 10,0.9703,0.6755,0.0000,0.3100,0.8046,0.007431
2,Antojitos Adriana,ANTOJITOS LUCY,Blvd. Tortuga 402,0.9324,0.6755,0.0000,0.7000,0.8052,0.016898
3,Tienda Lalo's,ABARROTES LALOS II,Fraccionamiento Geovillas los Pinos,0.9286,0.7141,0.0000,0.4300,0.8071,0.017854
4,Mini Super Fer,ABARROTES FER,"Hacienda, Hac. Virginia 18",0.9757,0.6755,0.0000,0.3100,0.8073,0.006082
5,Abarrotes DIAZ,ABARROTES EL SALVADOR,"Union Antorchista Vergara Tarimoya4, Fraternidad Mz83 LT32",0.9328,0.6755,0.0000,0.7400,0.8074,0.016791
6,Michi´s Burger,BURGERS MICHIS,C. 11 entre Avenida 3 y 4,0.9076,0.6755,0.0000,1.0000,0.8078,0.023089
7,Aba. La higuera,HCV HIGUERA,P.º dos Valles 96,0.9580,0.6755,0.0000,0.5000,0.8080,0.010493
8,Abarrotes Mota,ABARROTES B&C,Montura 27,0.9333,0.6755,0.0000,0.7500,0.8081,0.016682
9,MINI SUPER LOS PINITOS,SUPER CENTRO LOS PINITOS,"Pelícano, Fraccionamiento Geovillas los Pinos",0.9600,0.6755,0.0000,0.5000,0.8090,0.009988


,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,Abarrotes Yenny,ABARROTES SANTY,C. Río Lorez 198,0.8345,0.6755,0.0000,0.5900,0.7507,0.041375
1,abarrotes cristy,ABARROTES 5 ETAPA,C. Río Besaya 175,0.8996,0.5869,0.0000,0.7400,0.7509,0.025090
2,Carnitas Rana Mix,CARNITAS RANAMIX TEJERIA,Av. Hipólito Saldaña 265,0.9624,0.5100,0.0000,0.8100,0.7512,0.009408
3,Abarrotes jazmín,ABARROTES ZOE,C. Rio Iguazu 106,0.8149,0.6755,0.0000,0.8200,0.7524,0.046271
4,Carnicería San Agustín,AB SN AGUSTIN,Granizo 4,0.9755,0.5100,0.0000,0.7200,0.7532,0.006118
5,Vig's Pollo,VIGS POLLOS RIO MEDIO 1,Fraccionamiento Lomas del Río Medio,0.9581,0.5260,0.0000,0.8000,0.7557,0.010479
6,Antojitos Georgina,ANTOJITOS REYNA,Del Árbol,0.8191,0.6755,0.0000,0.9000,0.7585,0.045231
7,Repostería Danyella & Krustaceo Yodan,ABARROTES DANYELLA,91809 Lomas de rio medio 4,0.9892,0.5260,0.0000,0.5700,0.7598,0.002699
8,Abarrotes Tamara,HCV TAMARA,San Rafael 76,0.8606,0.6755,0.0000,0.5300,0.7608,0.034841
9,ABARROTES 2 HERMANAS,ABARROTES LAS 2 HERMANAS,Limonero 421,0.5240,1.0000,0.0000,1.0000,0.7620,0.119009


...

,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,abarrotes MI TIENDA,ABARROTES EL MARIACHI,16 de Septiembre 2,0.5899,0.8219,0.0000,0.7100,0.7003,0.102515
1,La principal,AB LA LA PRINCIPAL II (ARA),P.º de Kukulcan 319,0.7762,0.6400,0.0000,0.6000,0.7061,0.055944
2,Pollería y Abarrotes Las Cuatas,ABARROTES VICTOR,C. Río Pisco 308,0.8162,0.5869,0.0000,0.7800,0.7112,0.045945
3,Abarrotes dominic,ABARROTES JOSE,"Condominio, Priv. Nicolas Bravo numero 14",0.7544,0.6755,0.0000,0.6700,0.7146,0.061406
4,Carnitas 2 Hermanos,ABARROTES 2 HERMANOS,"Entrada casas ara, Azteca",0.6491,0.8063,0.0000,0.5500,0.7149,0.087733
5,abarrotes Hugo,ABARROTES TONY,Granate 70,0.7531,0.6755,0.0000,0.7500,0.7180,0.061715
6,Abarrotes Rosy,HCV ROSY 2,San Cipriano 157-267,0.8632,0.5869,0.0000,0.4600,0.7187,0.034205
7,abarrotes la roka,ABARROTES POOT,Marcelino Fernandez Lavid,0.7578,0.6755,0.0000,0.7500,0.7204,0.060551
8,Zona de Cantinas y Carnitas,CARNITAS Y CHICHARRONES PORKY,Av. Úrsulo Galván 776,0.9202,0.5100,0.0000,0.6200,0.7206,0.019951
9,Abarrotes lucero,LUCERO ARELI VAZQUES CRUZ,Zapote 26,0.9166,0.5260,0.0000,0.5600,0.7230,0.020860


,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,abarrotes MI TIENDA,ABARROTES EL MARIACHI,16 de Septiembre 2,0.5899,0.8219,0.0000,0.7100,0.7003,0.102515
1,La principal,AB LA LA PRINCIPAL II (ARA),P.º de Kukulcan 319,0.7762,0.6400,0.0000,0.6000,0.7061,0.055944
2,Pollería y Abarrotes Las Cuatas,ABARROTES VICTOR,C. Río Pisco 308,0.8162,0.5869,0.0000,0.7800,0.7112,0.045945
3,Abarrotes dominic,ABARROTES JOSE,"Condominio, Priv. Nicolas Bravo numero 14",0.7544,0.6755,0.0000,0.6700,0.7146,0.061406
4,Carnitas 2 Hermanos,ABARROTES 2 HERMANOS,"Entrada casas ara, Azteca",0.6491,0.8063,0.0000,0.5500,0.7149,0.087733
5,abarrotes Hugo,ABARROTES TONY,Granate 70,0.7531,0.6755,0.0000,0.7500,0.7180,0.061715
6,Abarrotes Rosy,HCV ROSY 2,San Cipriano 157-267,0.8632,0.5869,0.0000,0.4600,0.7187,0.034205
7,abarrotes la roka,ABARROTES POOT,Marcelino Fernandez Lavid,0.7578,0.6755,0.0000,0.7500,0.7204,0.060551
8,Zona de Cantinas y Carnitas,CARNITAS Y CHICHARRONES PORKY,Av. Úrsulo Galván 776,0.9202,0.5100,0.0000,0.6200,0.7206,0.019951
9,Abarrotes lucero,LUCERO ARELI VAZQUES CRUZ,Zapote 26,0.9166,0.5260,0.0000,0.5600,0.7230,0.020860


,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,abarrotes MI TIENDA,ABARROTES EL MARIACHI,16 de Septiembre 2,0.5899,0.8219,0.0000,0.7100,0.7003,0.102515
1,La principal,AB LA LA PRINCIPAL II (ARA),P.º de Kukulcan 319,0.7762,0.6400,0.0000,0.6000,0.7061,0.055944
2,Pollería y Abarrotes Las Cuatas,ABARROTES VICTOR,C. Río Pisco 308,0.8162,0.5869,0.0000,0.7800,0.7112,0.045945
3,Abarrotes dominic,ABARROTES JOSE,"Condominio, Priv. Nicolas Bravo numero 14",0.7544,0.6755,0.0000,0.6700,0.7146,0.061406
4,Carnitas 2 Hermanos,ABARROTES 2 HERMANOS,"Entrada casas ara, Azteca",0.6491,0.8063,0.0000,0.5500,0.7149,0.087733
5,abarrotes Hugo,ABARROTES TONY,Granate 70,0.7531,0.6755,0.0000,0.7500,0.7180,0.061715
6,Abarrotes Rosy,HCV ROSY 2,San Cipriano 157-267,0.8632,0.5869,0.0000,0.4600,0.7187,0.034205
7,abarrotes la roka,ABARROTES POOT,Marcelino Fernandez Lavid,0.7578,0.6755,0.0000,0.7500,0.7204,0.060551
8,Zona de Cantinas y Carnitas,CARNITAS Y CHICHARRONES PORKY,Av. Úrsulo Galván 776,0.9202,0.5100,0.0000,0.6200,0.7206,0.019951
9,Abarrotes lucero,LUCERO ARELI VAZQUES CRUZ,Zapote 26,0.9166,0.5260,0.0000,0.5600,0.7230,0.020860


,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,abarrotes MI TIENDA,ABARROTES EL MARIACHI,16 de Septiembre 2,0.5899,0.8219,0.0000,0.7100,0.7003,0.102515
1,La principal,AB LA LA PRINCIPAL II (ARA),P.º de Kukulcan 319,0.7762,0.6400,0.0000,0.6000,0.7061,0.055944
2,Pollería y Abarrotes Las Cuatas,ABARROTES VICTOR,C. Río Pisco 308,0.8162,0.5869,0.0000,0.7800,0.7112,0.045945
3,Abarrotes dominic,ABARROTES JOSE,"Condominio, Priv. Nicolas Bravo numero 14",0.7544,0.6755,0.0000,0.6700,0.7146,0.061406
4,Carnitas 2 Hermanos,ABARROTES 2 HERMANOS,"Entrada casas ara, Azteca",0.6491,0.8063,0.0000,0.5500,0.7149,0.087733
5,abarrotes Hugo,ABARROTES TONY,Granate 70,0.7531,0.6755,0.0000,0.7500,0.7180,0.061715
6,Abarrotes Rosy,HCV ROSY 2,San Cipriano 157-267,0.8632,0.5869,0.0000,0.4600,0.7187,0.034205
7,abarrotes la roka,ABARROTES POOT,Marcelino Fernandez Lavid,0.7578,0.6755,0.0000,0.7500,0.7204,0.060551
8,Zona de Cantinas y Carnitas,CARNITAS Y CHICHARRONES PORKY,Av. Úrsulo Galván 776,0.9202,0.5100,0.0000,0.6200,0.7206,0.019951
9,Abarrotes lucero,LUCERO ARELI VAZQUES CRUZ,Zapote 26,0.9166,0.5260,0.0000,0.5600,0.7230,0.020860


,dataplor_name,pos_name,dataplor_address,distance_score,name_score,address_score,phonetic_score,total_score,distance
0,abarrotes MI TIENDA,ABARROTES EL MARIACHI,16 de Septiembre 2,0.5899,0.8219,0.0000,0.7100,0.7003,0.102515
1,La principal,AB LA LA PRINCIPAL II (ARA),P.º de Kukulcan 319,0.7762,0.6400,0.0000,0.6000,0.7061,0.055944
2,Pollería y Abarrotes Las Cuatas,ABARROTES VICTOR,C. Río Pisco 308,0.8162,0.5869,0.0000,0.7800,0.7112,0.045945
3,Abarrotes dominic,ABARROTES JOSE,"Condominio, Priv. Nicolas Bravo numero 14",0.7544,0.6755,0.0000,0.6700,0.7146,0.061406
4,Carnitas 2 Hermanos,ABARROTES 2 HERMANOS,"Entrada casas ara, Azteca",0.6491,0.8063,0.0000,0.5500,0.7149,0.087733
5,abarrotes Hugo,ABARROTES TONY,Granate 70,0.7531,0.6755,0.0000,0.7500,0.7180,0.061715
6,Abarrotes Rosy,HCV ROSY 2,San Cipriano 157-267,0.8632,0.5869,0.0000,0.4600,0.7187,0.034205
7,abarrotes la roka,ABARROTES POOT,Marcelino Fernandez Lavid,0.7578,0.6755,0.0000,0.7500,0.7204,0.060551
8,Zona de Cantinas y Carnitas,CARNITAS Y CHICHARRONES PORKY,Av. Úrsulo Galván 776,0.9202,0.5100,0.0000,0.6200,0.7206,0.019951
9,Abarrotes lucero,LUCERO ARELI VAZQUES CRUZ,Zapote 26,0.9166,0.5260,0.0000,0.5600,0.7230,0.020860


In [10]:
# 0.80 is a safe thresholdfor supermatch 
# 0.60 is a safe thresholdfor nomatch

---
## 5. Set Your Threshold → Generate `super_match`

After reviewing the samples above, set `THRESHOLD` to the value where the 10 lowest records **still look like real matches**.  
`super_match = True` if `total_score >= THRESHOLD`, `False` otherwise (including rows with no score).

In [11]:
# ── ✏️  SET YOUR THRESHOLD HERE ───────────────────────────────────────────
THRESHOLD = 0.80
# ─────────────────────────────────────────────────────────────────────────

df['super_match'] = df['total_score'].apply(
    lambda s: True if pd.notna(s) and s >= THRESHOLD else False
)

n_true  = df['super_match'].sum()
n_false = (~df['super_match']).sum()
pct     = n_true / len(df) * 100

print(f'Threshold : {THRESHOLD}')
print(f'super_match = True  : {n_true:,}  ({pct:.1f}%)')
print(f'super_match = False : {n_false:,}  ({100-pct:.1f}%)')
print()
print('Preview (10 lowest True records):')

preview = (
    df[df['super_match'] == True]
    .nsmallest(10, 'total_score')[DISPLAY_COLS + ['super_match']]
    .reset_index(drop=True)
)

fmt = {c: '{:.4f}' for c in SCORE_COLS if c in preview.columns}
display(
    preview.style
    .format(fmt, na_rep='—')
    .background_gradient(subset=['total_score'], cmap='RdYlGn', vmin=0.40, vmax=0.95)
    .background_gradient(subset=['distance_score','name_score','address_score','phonetic_score'],
                         cmap='RdYlGn', vmin=0, vmax=1)
    .set_properties(**{'font-size':'11px','border':'1px solid #ddd','padding':'4px 8px'})
    .set_table_styles([{'selector':'th','props':[('background-color','#1a1a2e'),('color','white'),('font-size','11px'),('padding','6px 8px')]}])
)

Threshold : 0.8
super_match = True  : 2,724  (7.9%)
super_match = False : 31,541  (92.1%)

Preview (10 lowest True records):


,dataplor_name,pos_name,dataplor_address,pos_address,distance_score,name_score,address_score,phonetic_score,total_score,distance,super_match
0,Tacos Carmen,DEL CARMEN CERVANTES,"45590, San Antonio 238",SAN ANTONIO 238,0.9881,0.5980,0.8810,0.7400,0.8000,0.002984,True
1,La Casa del Waffle,LA CASA DEL WAFLE CHAPALITA,Av. Tepeyac 1037,AV TEPEYAC 1037,0.9719,0.5550,1.0000,0.7100,0.8001,0.007034,True
2,Pollo Pepe,POLLO PEPE MONUMENTAL,Av. Belisario Domínguez 1729,BELISARIO DOMINGUEZ 1729,0.9522,0.6550,1.0000,0.5900,0.8001,0.011960,True
3,Abarrotes Martha,MISCELANEA MARTHA,Los Belenes 1239 A,"BELENES 1239, A",0.9829,0.6850,1.0000,0.5000,0.8004,0.004281,True
4,Qin Oriental Food Centro,QIN CENTRO HISTORICO,C. Colón 37,"COLON 37, A",0.9915,0.6480,0.9230,0.6200,0.8004,0.002136,True
5,Frutas and Vegetables Guadalupe,FRUTAS Y VERDURAS GUADALUPE,Batalla de Zacatecas 10,"BATALLA DE ZACATECAS 10, A",0.7833,0.7250,1.0000,0.7400,0.8005,0.054184,True
6,Tacos de birria Don Luis,TACOS DE BIRRIA LUIS,Sta. Blanca 861-d,"AVENIDA ACUEDUCTO 861, D",0.9526,0.9230,0.3290,0.8600,0.8005,0.011850,True
7,Coco Cafe,COCO CAFE,Ramal de La Tijera 701,PRIV AZUCENA 6,0.8477,1.0000,0.2310,1.0000,0.8005,0.038076,True
8,Pollos Asados Country,ABARROTES COUNTRY,Donceles 2396,"DONCELES 2396, A",0.9444,0.5910,1.0000,0.7000,0.8006,0.013888,True
9,Cremeria La Noria,CREMERIA Y ABARROTES LA NORIA,C. Luis M. Fregoso 4476,LUIS M FREGOSO 147,0.9285,0.7180,0.8340,0.7000,0.8007,0.017885,True


## 6. Export Enriched CSV

In [12]:
out_path = os.path.join(NOTEBOOK_DIR, f'Datamatch_Femsa_RD_SuperMatch_thr{str(THRESHOLD).replace(".","_")}_20260626.csv')
df.to_csv(out_path, index=False)
print(f'✅ Saved: {os.path.basename(out_path)}')
print(f'   Rows: {len(df):,}   Columns: {df.shape[1]}   (super_match added)')

✅ Saved: Datamatch_Femsa_RD_SuperMatch_thr0_8_20260626.csv
   Rows: 34,265   Columns: 49   (super_match added)
